**1. CONVOLUTIONAL AUTOENCODERS TRAINING**

In [ ]:
import digitalhub as dh
import pandas as pd
import matplotlib.pyplot as plt

NOME_PROGETTO = "floods"
project = dh.get_project(NOME_PROGETTO)
print(f"Progetto: {project.name}")

**SETUP PARAMETERS**

In [ ]:
# Parametri Job
job_name = "train_s2_v3"                                    
dataset = "Standard" 
epochs = 200
train_sar = False
train_opt = True                                           # Test, Standard, Anomalies*
#handler = pretrain_encoders                                         

parametri = {
    "epochs": epochs, 
    "batch_size": 16, 
    "lr": 1e-4, 
    "weight_decay": 1e-4,      
    "patch_size": 256, 
    "n_images1": 4, "n_channels1": 2,                       # sar
    "n_images2": 4, "n_channels2": 10,                      # opt
    "output_dim": 512,                                      # dimensione spazio latente                                                     
    "mamba": False, 
    "workers": 0,
    "job_name": job_name,
    "dataset": dataset,
    "train_sar": train_sar,                                  # train sar encoder
    "train_opt": train_opt,                                  # train opt encoder
    "patience": 20,                                    
    "min_delta": 1e-4,
    "time_debug": False                                     # time_debug = True solo per debug, = False per training
}

print(f"PARAMETRI: {parametri}")

# volume -> circa 400 GB dataset Standard
volumi = [
    {
        "volume_type": "ephemeral",
        "name": "volume-spazio-dati",
        "mount_path": "/data",      
        "spec": {"size": "500Gi"}   
    }
]

**BUILD ENVIRONMENT**

In [ ]:
encoders_train_func = project.new_function(
    name= f'encoders-Floods_{job_name}_{dataset}_{epochs}',
    kind="python",
    python_version="PYTHON3_10",
    code_src="../src/", 
    handler="train_autoencoders_1D", 
    base_image="pytorch/pytorch:2.1.2-cuda11.8-cudnn8-runtime",
    requirements=["pandas==2.3.3", "numpy==1.26.4", "rasterio==1.4.4", "tqdm==4.70.0", "tifffile==2024.8.30"]
)

# .run("build") -> scarica immagine, installa requirements, copia intera cartella code_src, crea immagine docker

build = encoders_train_func.run("build", wait=True)
print(f"BUILD: {build.status.state}")

**TRAINING**

In [ ]:
# action job = avvia container, esegue script, libera risorse

run_train_encoders = encoders_train_func.run(
    action="job", 
    parameters=parametri, 
    volumes=volumi, 
    profile="1xV100",                                       # 1x = 1 gpu
    # local_execution= True,                                
    wait=True
)

print(f"Run train_encoders avviato: {run_train_encoders.id}")
print(run_train_encoders.status.state)
print(run_train_encoders.status.message)

**PLOTS**

In [ ]:
if train_sar:
    # salvataggio log
    path_s1 = project.get_artifact(f"metrics-s1_{job_name}_{dataset}_{epochs}").download(overwrite=True)
    df_s1 = pd.read_csv(path_s1)

    # plot
    plt.figure(figsize=(8, 5))
    plt.plot(df_s1['epoch'], df_s1['train_loss'], color='blue', label='Train Loss')
    plt.title(f'Training CAE SAR - {job_name}_{dataset}')
    plt.xlabel('Epochs')
    plt.ylabel('Loss (MSE)')
    plt.grid(True)
    plt.legend()
    plt.show()

else:
    print("train sar non eseguito")

In [ ]:
if train_opt:
    # salvataggio log
    path_s2 = project.get_artifact(f"metrics-s2_{job_name}_{dataset}_{epochs}").download(overwrite=True)
    df_s2 = pd.read_csv(path_s2)

    # plot
    plt.figure(figsize=(8, 5))
    plt.plot(df_s2['epoch'], df_s2['train_loss'], color='red', label='Train Loss')
    plt.title(f'Training CAE OPT - {job_name}_{dataset}_{epochs}')
    plt.xlabel('Epochs')
    plt.ylabel('Loss (MSE)')
    plt.grid(True)
    plt.legend()
    plt.show()

else:
    print("train opt non eseguito")